# Viva voice job: WAAD_patch2

Runs end to end with **Run all** (T4 GPU). Uploads `WAAD_patch2.zip` to the `voice-work` release of the waad repo (needs the Colab secret `GH_TOKEN`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q chatterbox-tts openai-whisper num2words requests
!pip install -q torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
import os
OUT = '/content/WAAD_patch2'; os.makedirs(OUT + '/best', exist_ok=True)
!wget -q -O /content/viva_ref.wav https://raw.githubusercontent.com/aseembhalla/waad-media/main/media/voice/viva_heart_ref.wav
print('ready')

In [ ]:
LINES = [
{
"ch": 5,
"line": 1,
"seg": 0,
"text": "Democrats flipped at least forty-four Republican House seats, and eight Republican Senate seats.",
"ex": 0.5,
"cfg": 0.5
},
{
"ch": 5,
"line": 6,
"seg": 0,
"text": "The final seat count isn't certified yet, so we'll stick with \"at least forty-four. \"",
"ex": 0.5,
"cfg": 0.5
}
]

In [ ]:
import re, json, difflib, torch, torchaudio as ta, whisper
from num2words import num2words
from chatterbox.tts import ChatterboxTTS
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = ChatterboxTTS.from_pretrained(device=dev)
asr = whisper.load_model('small.en', device=dev)
REF, TAKES = '/content/viva_ref.wav', 4
def num(m):
    n = int(m.group(0))
    return num2words(n, to='year') if 1800 <= n <= 2099 and len(m.group(0)) == 4 else num2words(n)
def norm(s):
    s = s.lower().replace('%', ' percent').replace('$', ' ')
    s = re.sub(r'(\d+)\.(\d+)', lambda m: m.group(1) + ' point ' + ' '.join(m.group(2)), s.replace(',', ''))
    s = re.sub(r'\d+', num, s).replace('dollars', ' ').replace('-', ' ').replace('waad', 'wad').replace('wad s', 'wads')
    return re.sub(r"[^a-z' ]", ' ', s).split()
def score(text, path):
    heard = asr.transcribe(path, language='en', fp16=(dev == 'cuda'))['text']
    return difflib.SequenceMatcher(None, norm(text), norm(heard)).ratio(), heard.strip()
report = []
for k, L in enumerate(LINES):
    cands = []
    for t in range(TAKES):
        wav = tts.generate(L['text'], audio_prompt_path=REF, exaggeration=L['ex'], cfg_weight=L['cfg'])
        p = '/content/_take.wav'; ta.save(p, wav, tts.sr)
        s, heard = score(L['text'], p); cands.append((s, t, wav, heard))
        if s >= 0.98: break
    s, t, wav, heard = max(cands, key=lambda c: c[0])
    ta.save(f"{OUT}/best/{L['ch']}_{L['line']:02d}_{L['seg']}.wav", wav, tts.sr)
    flag = '' if s >= 0.85 else '   <-- CHECK'
    report.append(f"[{L['ch']}.{L['line']}.{L['seg']}] match {s:.2f} ({len(cands)} takes){flag}\n   script: {L['text']}\n   heard:  {heard}")
    print(f'{k+1}/{len(LINES)}  match {s:.2f}{flag}')
json.dump(LINES, open(f'{OUT}/segments.json', 'w'), indent=1)
open(f'{OUT}/report.txt', 'w').write('\n'.join(report))
print('Done')

In [ ]:
import requests
from google.colab import userdata
!cd /content && rm -f WAAD_patch2.zip && zip -qr WAAD_patch2.zip WAAD_patch2
tok = userdata.get('GH_TOKEN')
H = {'Authorization': f'Bearer {tok}', 'Accept': 'application/vnd.github+json'}
R = 'https://api.github.com/repos/aseembhalla/waad'
r = requests.get(f'{R}/releases/tags/voice-work', headers=H)
if r.status_code == 404:
    r = requests.post(f'{R}/releases', headers=H, json={'tag_name': 'voice-work', 'name': 'Voice work (temporary uploads from Colab)',
        'body': 'Temporary audio from Colab voice jobs; replaced on every upload.', 'prerelease': True})
rel = r.json()
for a in rel.get('assets', []):
    requests.delete(f"{R}/releases/assets/{a['id']}", headers=H)
up = requests.post(f"https://uploads.github.com/repos/aseembhalla/waad/releases/{rel['id']}/assets?name=WAAD_patch2.zip",
                   headers={**H, 'Content-Type': 'application/zip'}, data=open('/content/WAAD_patch2.zip', 'rb'))
print('UPLOAD', up.status_code, up.json().get('browser_download_url', up.text[:200]))